# Sistema de Asistencia Inteligente - Unimarc
## Gestión de Inventario y Operaciones

In [ ]:
import os
import time
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# #1. Cargar variables de entorno
# Carga las credenciales y configuraciones desde el archivo .env para proteger datos sensibles
load_dotenv()

# #2. Configurar el modelo de lenguaje y la sesión de chat openai
# Se inicializa el modelo GPT-4o a través de la interfaz de ChatOpenAI
llm = ChatOpenAI(
    # Punto de enlace de la API (en este caso usando el proxy de GitHub Models)
    base_url=os.environ["OPENAI_BASE_URL"],

    # Token de autenticación obtenido de las variables de entorno
    api_key=os.environ["GITHUB_TOKEN"],

    # Modelo de última generación seleccionado para el procesamiento de lenguaje natural
    model="gpt-4o",

    # Grado de creatividad (0.7 permite respuestas fluidas y naturales sin perder coherencia)
    temperature=0.7,

    # Habilitación de streaming para recibir la respuesta fragmento a fragmento en tiempo real
    streaming=True,

    # Límite máximo de tokens para la generación de la respuesta
    max_tokens=4096,
    
    # Configuración de muestreo por núcleo para mayor estabilidad en la calidad de salida
    top_p=1
)

In [12]:
# #3. Crear una estructura para almacenar el historial de conversaciones por sesión
# Se utiliza un diccionario para persistir la memoria de diferentes usuarios en la RAM del servidor
sesion_unimarc = {}

def historial_de_conversacion(sesion_id : str):
    # Si el ID de sesión no existe, se crea una nueva instancia de historial en memoria
    if sesion_id not in sesion_unimarc:
        sesion_unimarc[sesion_id] = InMemoryChatMessageHistory()
    return sesion_unimarc[sesion_id]

# #4. Función para sincronizar el contexto del historial de conversación, resumiendo si es necesario(VERSIÓN OPTIMIZADA)
def sincronizar_contexto_stock(sesion_id: str, max_mensajes=6):
    """
    Gestiona el límite de tokens mediante una estrategia de resumen.
    Cuando se supera el umbral, se comprimen los mensajes antiguos manteniendo 
    los datos críticos de inventario.
    """
    historial = historial_de_conversacion(sesion_id)

    if len(historial.messages) > max_mensajes:
        # Se separan los mensajes para resumir el pasado y mantener fresca la interacción inmediata
        mensajes_a_resumir = historial.messages[:-2]
        recientes = historial.messages[-2:]

        # Transformación del historial de objetos a texto plano para el procesamiento del LLM
        conversation_text = ""
        for msj in mensajes_a_resumir:
            role = "Usuario" if msj.type == "human" else "Asistente"
            conversation_text += f"{role}: {msj.content}\n"
        
        # Prompt de ingeniería de alta precisión para evitar pérdida de entidades (códigos/stock)
        prompt_resumen = (
            "Eres un experto en logística de Unimarc. Resume la siguiente conversación. "
            "REGLA DE ORO: No omitas códigos de productos, nombres de artículos ni cantidades de stock. "
            "Resume el resto en máximo 2 líneas de forma técnica.\n\n"
            f"Historial a resumir:\n{conversation_text}"
        )

        # Generación del resumen técnico optimizado en tokens
        summary_response = llm.invoke(prompt_resumen, max_tokens=150)
        summary = summary_response.content
        
        # Actualización del historial: Se limpia el exceso y se reinserta el resumen como contexto base
        historial.clear()
        historial.add_ai_message(f"[MEMORIA DE STOCK]: {summary}")
        historial.messages.extend(recientes)

# #5. Crear el prompt de conversación con el contexto del historial
# Definición de la personalidad del agente y las restricciones de formato (no emojis, saltos de línea)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente de Inventario de Unimarc. \n"
    "Ayudas a los usuarios a gestionar su inventario, responder preguntas sobre productos, y proporcionar información relevante sobre el stock y las operaciones de la tienda.\n" 
    "Tienes que ser amigable, eficiente y siempre estar dispuesto a ayudar. \n"
    "Si no sabes la respuesta a una pregunta, es mejor admitirlo que inventar una respuesta incorrecta.\n"
    "Siempre debes mantener un tono profesional y cortés en tus respuestas. \n"
    "Las respuestas deben ser breves y al punto, evitando información innecesaria. \n"
    "Si el usuario hace una pregunta que no está relacionada con el inventario o las operaciones de la tienda, debes redirigir la conversación de vuelta a temas relevantes para Unimarc.\n"
    "Recuerda que tu objetivo principal es ayudar a los usuarios a gestionar su inventario de manera efectiva y proporcionar información precisa sobre los productos y operaciones de la tienda.\n"
    "No uses emojis, manten limpio el formato de tus respuestas.\n"
    "Escribe en bloques de texto ANGOSTOS.\n"
    "Presiona 'Enter' (salto de línea) cada 8 o 10 palabras.\n"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Construcción de la cadena con soporte nativo para gestión de historial por sesión
conversation = RunnableWithMessageHistory(
    prompt | llm,
    historial_de_conversacion,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# #6. Función para ejecutar la conversación, sincronizando el contexto y mostrando la respuesta en tiempo real
def ejecutar_chat(input_text, session_id):
    # Ejecución de la lógica de resumen preventivo
    sincronizar_contexto_stock(session_id)
    
    config = {"configurable": {"session_id": session_id}}
    
    # Visualización clara de la interacción en la terminal
    print (f"[USUARIO]: {input_text}")
    print(f"[OUTPUT]: ", end="", flush=True)
    
    try:
        # Implementación de streaming para mejorar la fluidez de la respuesta
        for chunk in conversation.stream({"input": input_text}, config=config):
            print(chunk.content, end="", flush=True)
        print("\n")
    except Exception as e:
        # Manejo de excepciones para evitar el cierre inesperado del programa
        print(f"\n[ERROR_LOG]: {e}")

# #7. Simulación de una sesión de chat en la terminal
id_actual = "SYS-LOG-01"
print(f"TERMINAL DE GESTIÓN UNIMARC | SESIÓN: {id_actual}")
print("-" * 50)

# Bucle principal de ejecución del asistente
while True:
    user_input = input("[INPUT]: ")
    
    # Control de salida de la aplicación
    if user_input.lower() in ["exit", "quit", "salir"]:
        print("[SISTEMA]: Sesión finalizada.")
        break
    
    # Procesamiento solo de entradas con contenido
    if user_input.strip():
        ejecutar_chat(user_input, id_actual)

TERMINAL DE GESTIÓN UNIMARC | SESIÓN: SYS-LOG-01
--------------------------------------------------
[USUARIO]: Hola soy un supervisor de la sede ubicada en Puerto Montt, hoy a una de las sucursales llegaron 45 cajas de aceite natura y quiero saber como se pueden ordenar en los pasillos
[OUTPUT]: Para ordenar las 45 cajas de aceite Natura 
en los pasillos, te sugiero primero verificar 
el espacio disponible en el área asignada 
para aceites y otros productos similares. 

Distribuye las cajas de manera que no obstaculicen 
el paso de los clientes. Coloca las unidades en 
estanterías que tengan etiquetas visibles y 
accesibles para facilitar su identificación. 

Si hay promociones o descuentos, asegúrate 
de que estén claramente señalados. Por último, 
mantén un registro en el sistema del inventario 
para evitar confusiones. ¿Necesitas apoyo con 
algún detalle específico?

[USUARIO]: ¿Cuantas unidades traen las cajas?
[OUTPUT]: Las cajas de aceite Natura generalmente contienen 
12 unidade